In [0]:
# Create openAI client
client = OpenAI(
    api_key = "YOUR_DATABRICKS_ACCESS_TOKEN",
    base_url = "https://adb-1884344040313130.10.azuredatabricks.net/"
)

In [0]:
# Sending simple chat request to LLM
# OpenAI request
completion = client.chat.completions.create(
    model = "databricks-claude-sonnet-4-6",
    messages = [
        {
            "role": "system", 
            "content": [
                {
                    "type": "text",
                    "text": "You are Batman, the protector of Gotham City"
                }
            ]
        },
        {"role": "user", 
         "content": [
             {
                 "type": "text",
                 "text": "How's Gotham City doing today?"
             }
         ]
        }
    ],
    temperature = 0.0,
    max_tokens = 1034
)
# print the respo

**Generating a custom python funtion using MLflow that can be served as a mosaic AI endpoint**

In [0]:
# import libraries
import mlflow
from mlflow import pyfunc

class BasicChatBot(pyfunc.PythonModel):
    def __init__(self, model_name: str):
        self.model_name = model_name
    
    def chatCompletionAPI(self, user_query):
        openai_client = OpenAI(
            api_key = "YOUR_DATABRICKS_ACCESS_TOKEN",
            base_url = "https://adb-1884344040313130.10.azuredatabricks.net/"
        )

        response = openai_client.chat.completions.create(
            model = self.model_name,
            messages = [
                {
                    "role": "system",
                    "content": [
                        {
                            "type": "text",
                            "text": "You are Batman, the protector of Gotham City"
                        }
                    ]
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": user_query
                        }
                    ]
                }
            ],
            temperature = 0.7 
        )
        return response.choices[0].message.content
    
    def predict(self, context, data):
        user_query = data["user_query"].iloc[0]
        gpt_response = self.chatCompletionAPI(user_query)

        return gpt_response

In [0]:
# Test the model 
test_model = BasicChatBot(model_name = "databricks-claude-sonnet-4-6")

In [0]:
from mlflow.models import infer_signature
import pandas as pd

# sample input
input_example = pd.DataFrame([{
    "user_query": "How's Gotham City doing today?"
}])

# Sample output (what the model actually returns)
output_example = pd.DataFrame({[]
    "predictions": "Gothom City is facing cloudy weather rising tensions downtown"
}])


# Infer full signature (input + output)
signature = infer_signature(input_example, output_example)
model_path = "basicchatbot"

mlflow.pyfunc.save_model(
    path = model_path,
    python_model=test_model,
    signature=signature,
    input_example=input_example
)
 

In [0]:
# Load the saved model 
loaded_pyfunc_model = mlflow.pyfunc.load_model(model_path)

# Test the loaded model 
model_input = pd.DataFrame([{
    "user_query": "Hello, how are you today?"
}])

model_response = loaded_pyfunc_model.predict(model_input)

print(model_response)

In [0]:
#Logging our Saved/Loaded Model
run_id = None

#Logg the model as an artifact
with mlflow.start_run() as run:
    mlflow.log_artifact(local_dir = model_path, artifact_path="BasicChatBot")
    run_id = run.info.run_id
    print(f"Model logged with run Id: {run_id}")
    

In [0]:
# Registering the model in Unity Catalog from the run Id
mlflow.register_model(f"runs:/{run_id}/BasicChatBot", "BasicChatBot")

In [0]:
# Testing the real-time endpoint: Use the following sample payload when testing the endpoint from the UI


{
  "dataframe_split": {
    "columns": [
      "user_query"
    ],
    "data": [
      [
        "How is Addis Ababa City doing today?"
      ]
    ]
  }
}